# Test `smutuvi/gemma-4-e2b-sw-asr-ndizi` on Google Colab

LoRA adapter for Swahili ASR on [`google/gemma-4-E2B-it`](https://huggingface.co/google/gemma-4-E2B-it).

**Before running:**
1. Runtime → **Change runtime type** → **GPU** (T4 or better).
2. Accept the Gemma license on Hugging Face for `google/gemma-4-E2B-it`.
3. Create an HF token with **read** access: https://huggingface.co/settings/tokens

## 1. Install dependencies

In [ ]:
!pip install -q "transformers>=5.0" "peft>=0.10" "accelerate>=0.30" "datasets>=2.18" torch torchaudio soundfile

## 2. Hugging Face login

In [ ]:
from huggingface_hub import login

# Paste token when prompted, or set HF_TOKEN in Colab secrets and uncomment:
# from google.colab import userdata
# login(token=userdata.get("HF_TOKEN"))
login()

## 3. Configuration

In [ ]:
BASE_MODEL = "google/gemma-4-E2B-it"
ADAPTER_REPO = "smutuvi/gemma-4-e2b-sw-asr-ndizi"

# Set True only if the Hub repo contains merged full weights (not a PEFT adapter).
USE_MERGED_REPO = False

# Set True on T4 if bf16 OOM; adapter still loads on 4-bit base.
USE_4BIT_BASE = False

TARGET_SR = 16_000
MAX_AUDIO_SEC = 30.0

ASR_INSTRUCTION = (
    "Transcribe the following speech segment in Swahili into Swahili text.\n\n"
    "Follow these specific instructions for formatting the answer:\n"
    "* Only output the transcription, with no newlines.\n"
    "* When transcribing numbers, write the digits, i.e. write 1.7 and not "
    "one point seven, and write 3 instead of three."
)

## 4. Load model

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

dtype = torch.bfloat16
load_kw = dict(dtype=dtype, device_map="auto", attn_implementation="sdpa")
if USE_4BIT_BASE:
    load_kw = dict(
        device_map="auto",
        attn_implementation="sdpa",
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        ),
    )

if USE_MERGED_REPO:
    processor = AutoProcessor.from_pretrained(ADAPTER_REPO, padding_side="left")
    model = AutoModelForMultimodalLM.from_pretrained(ADAPTER_REPO, **load_kw).eval()
    print(f"Loaded merged model from {ADAPTER_REPO}")
else:
    processor = AutoProcessor.from_pretrained(BASE_MODEL, padding_side="left")
    base = AutoModelForMultimodalLM.from_pretrained(BASE_MODEL, **load_kw)
    model = PeftModel.from_pretrained(base, ADAPTER_REPO).eval()
    print(f"Loaded {ADAPTER_REPO} adapter on {BASE_MODEL}")

print("Device:", next(model.parameters()).device)

## 5. Inference helpers (matches `ndizi_mlops_gemma-4` training recipe)

In [ ]:
import numpy as np


def prepare_audio(wave, sr):
    wave = np.asarray(wave, dtype=np.float32)
    if wave.ndim > 1:
        wave = wave.mean(axis=-1)
    if int(sr) != TARGET_SR:
        import torchaudio.functional as taf

        wave = taf.resample(torch.from_numpy(wave), int(sr), TARGET_SR).numpy().astype(np.float32)
    max_samples = int(MAX_AUDIO_SEC * TARGET_SR)
    if len(wave) > max_samples:
        wave = wave[:max_samples]
    return wave


def gemma_soft_token_count(processor, fe_out):
    mask = fe_out["input_features_mask"][0]
    if hasattr(mask, "numpy"):
        mask = mask.numpy()
    num_mel = int(np.asarray(mask, dtype=bool).sum())
    if num_mel <= 0:
        return 0
    t = num_mel
    for _ in range(2):
        t = (t + 2 - 3) // 2 + 1
    return min(t, processor.audio_seq_length)


def build_inputs(processor, wave, instruction=ASR_INSTRUCTION):
    fe_out = processor.feature_extractor([wave], return_tensors="pt")
    n_soft = gemma_soft_token_count(processor, fe_out)
    boa, at, eoa = processor.boa_token, processor.audio_token, processor.eoa_token
    audio_block = f"{boa}{at * n_soft}{eoa}"

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio": wave},
                {"type": "text", "text": instruction},
            ],
        }
    ]
    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    if isinstance(prompt, list):
        prompt = prompt[0]
    if at not in prompt:
        raise ValueError("Chat template missing audio placeholder token")
    prompt = prompt.replace(at, audio_block, 1)

    inputs = processor(text=prompt, return_tensors="pt", return_mm_token_type_ids=True)
    inputs["input_features"] = fe_out["input_features"]
    inputs["input_features_mask"] = fe_out["input_features_mask"]
    return inputs


def inputs_to_device(inputs, model):
    inputs = inputs.to(model.device)
    for key, val in inputs.items():
        if key in ("input_features", "input_features_mask"):
            continue
        if isinstance(val, torch.Tensor) and val.is_floating_point():
            inputs[key] = val.to(dtype=model.dtype)
    return inputs


def transcribe(model, processor, wave, sr, instruction=ASR_INSTRUCTION, max_new_tokens=256):
    wave = prepare_audio(wave, sr)
    inputs = build_inputs(processor, wave, instruction=instruction)
    inputs = inputs_to_device(inputs, model)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[:, inputs["input_ids"].shape[-1] :]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0]

## 6. Test: upload a WAV file

In [ ]:
from google.colab import files
import soundfile as sf

uploaded = files.upload()
path = next(iter(uploaded))
wave, sr = sf.read(path, dtype="float32", always_2d=True)
wave = wave.mean(axis=1)

hyp = transcribe(model, processor, wave, sr)
print("File:", path)
print("Duration (s):", len(wave) / sr)
print("--- transcription ---")
print(hyp)

## 7. Test: sample from `smutuvi/ndizi-1` (optional)

In [ ]:
from datasets import load_dataset

ds = load_dataset("smutuvi/ndizi-1", split="test", streaming=True)
row = next(iter(ds))
a = row["audio"]
ref = row.get("text") or row.get("sentence") or ""

hyp = transcribe(model, processor, a["array"], a["sampling_rate"])
print("reference:", ref)
print("hypothesis:", hyp)

## 8. Compare to zero-shot base (optional; needs extra VRAM)

In [ ]:
RUN_BASELINE_COMPARE = False  # set True to run

if RUN_BASELINE_COMPARE:
    del model
    import gc

    gc.collect()
    torch.cuda.empty_cache()

    processor_b = AutoProcessor.from_pretrained(BASE_MODEL, padding_side="left")
    base_model = AutoModelForMultimodalLM.from_pretrained(BASE_MODEL, **load_kw).eval()

    print("baseline:", transcribe(base_model, processor_b, wave, sr))

    del base_model
    gc.collect()
    torch.cuda.empty_cache()

    processor = AutoProcessor.from_pretrained(BASE_MODEL, padding_side="left")
    base = AutoModelForMultimodalLM.from_pretrained(BASE_MODEL, **load_kw)
    model = PeftModel.from_pretrained(base, ADAPTER_REPO).eval()
    print("finetuned:", transcribe(model, processor, wave, sr))